# 14 — Dynamic Window Approach (DWA)

**Section:** Motion Planning · **Mirrors MATLAB:** *Path Following with Obstacle Avoidance*

DWA is a **local** planner: at each control step it samples reachable $(v, \omega)$ pairs, simulates each forward for a short horizon, and scores the resulting trajectory.


## Intuition — what's actually going on?

A\* and RRT plan a path from where you are to where you want to go *upfront* — but they assume the world is static. In reality, obstacles move, the map is uncertain, and you only have control over the next fraction of a second of motion. That's where **local planners** come in.

The Dynamic Window Approach (DWA) thinks like this: "I'm a wheelchair-like robot. In the next 0.1 seconds I can only change my speed and turn rate by a limited amount (because of my motor accelerations). So let me **sample** a bunch of plausible (speed, turn-rate) pairs in that *dynamic window*, simulate each one forward for ~2 seconds with the current world, and pick whichever one looks best."

"Best" is a hand-tuned weighted sum: close to the goal, far from obstacles, fast forward. It's a single-step receding-horizon planner — very cheap to compute, very reactive to new obstacles. The animation in the README shows it gracefully detouring around dynamic obstacles.

DWA's blind spot: it's myopic. It can't reason about narrow gaps several seconds out — it'll happily steer into a dead-end alley. Combine DWA with a global planner (A\*, RRT) to fix that.


## Analytical derivation

**Dynamic window.** For a differential-drive robot with bounded velocity and acceleration, the set of *admissible* controls in the next step $\Delta t$ is

$$V_d = \bigl\{(v, \omega) \;:\; v \in [\max(v_\min, v - a_\max \Delta t),\ \min(v_\max, v + a_\max \Delta t)],\ \omega \in [\max(\omega_\min, \omega - \alpha_\max \Delta t),\ \min(\omega_\max, \omega + \alpha_\max \Delta t)]\bigr\}$$

This is the "dynamic window" — a rectangle in $(v, \omega)$-space centered at the current velocity, sized by acceleration limits.

**Trajectory rollout.** For each $(v, \omega) \in V_d$, simulate forward for horizon $T_p$ using the unicycle model:

$$x(\tau) = x_0 + v \cos\theta_0\,\tau,\quad y(\tau) = y_0 + v \sin\theta_0\,\tau,\quad \theta(\tau) = \theta_0 + \omega \tau$$

(For constant $(v,\omega)$ this is actually an arc, but discretized as a sequence of points.)

**Cost function.** Pick the $(v^*, \omega^*) \in V_d$ minimizing

$$J(v, \omega) \;=\; w_g\,\|\mathbf{p}_T - \mathbf{p}_\text{goal}\|\;+\;w_h\,|\theta_T - \theta_\text{goal-direction}|\;+\;\frac{w_o}{d_\min}\;+\;w_v\,(v_\max - v_T)$$

with $\mathbf{p}_T = (x(T_p), y(T_p))$, $d_\min$ the minimum clearance to any obstacle over the trajectory. Hard constraint: $J = +\infty$ if $d_\min < r_\text{safety}$.

Each term has a clear role:
- **Goal-distance term** ($w_g$) pulls the robot toward the goal.
- **Heading term** ($w_h$) penalizes pointing the wrong way at end-of-horizon.
- **Obstacle clearance term** ($w_o / d_\min$) is large when we get near an obstacle.
- **Velocity reward** ($w_v$) prefers faster motion to avoid getting stuck.

The result is a *receding-horizon, sample-based, single-step MPC* — much cheaper than full MPC but myopic (no look-ahead beyond $T_p$).

### Compatibility check — math ↔ code

| Step | Code |
|---|---|
| Dynamic window in $v$ | `vs = np.linspace(max(v_min, v - a_max*dt), min(v_max, v + a_max*dt), 7)` |
| Dynamic window in $\omega$ | `ws = np.linspace(max(w_min, w - alpha_max*dt), min(w_max, w + alpha_max*dt), 11)` |
| Trajectory rollout for $T_p$ s | `for _ in range(int(predict_t/dt)): s[0]+=v*cos*dt; s[1]+=v*sin*dt; s[2]+=w*dt` |
| Goal-distance term $w_g\|p_T - p_g\|$ | `1.0 * goal_dist` |
| Heading term $w_h\|\theta_T - \angle(p_g - p_T)\|$ | `0.3 * heading_cost` where `heading_cost = abs(np.arctan2(dy,dx) - traj[-1,2])` |
| Clearance term $w_o / d_\min$ | `0.2 / d.min()` |
| Hard clearance constraint | `if d.min() < 0.5: return np.inf` |
| Velocity reward $w_v(v_\max - v_T)$ | `0.05 * (v_max - traj[-1, 3])` |
| Pick best $(v^*,\omega^*)$ | nested loop `if c < best_c: best_c, best_v, best_w = c, v, w` |


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

robot = np.array([0., 0., 0., 0., 0.])      # x, y, theta, v, omega
goal  = np.array([10., 10.])
obstacles = np.array([[3, 4], [5, 6], [7, 8], [4, 7], [6, 5], [8, 9]])

v_max, v_min = 1.0, 0.0
w_max, w_min = 1.0, -1.0
a_max, alpha_max = 4.0, 6.0      # window must be larger than discrete step
dt, predict_t = 0.1, 2.5


In [ ]:
def predict(state, v, w):
    s = state.copy()
    traj = [s.copy()]
    for _ in range(int(predict_t / dt)):
        s[0] += v * np.cos(s[2]) * dt
        s[1] += v * np.sin(s[2]) * dt
        s[2] += w * dt
        s[3], s[4] = v, w
        traj.append(s.copy())
    return np.array(traj)


# COUNCIL FIX (pass 14, Morari): name the safety margin; document the safety set.
r_safety = 0.5     # robot radius + safety margin (m). Trajectory rejected if any
                   # waypoint comes within r_safety of any obstacle centre.


def cost(traj, goal, obstacles):
    dx, dy = goal - traj[-1, :2]
    goal_dist    = np.hypot(dx, dy)
    heading_cost = abs(np.arctan2(dy, dx) - traj[-1, 2])
    d = np.min(np.linalg.norm(traj[:, None, :2] - obstacles[None], axis=2), axis=1)
    if d.min() < r_safety:
        return np.inf
    # Weights tuned to this scene; council fix (Hespanha): not transferable
    # across environments. Production stacks tune per-env or use inverse RL
    # on demonstration trajectories.
    return 1.0 * goal_dist + 0.3 * heading_cost + 0.2 / d.min() + 0.05 * (v_max - traj[-1, 3])


In [ ]:
hist = [robot[:2].copy()]
for _ in range(200):
    vs = np.linspace(max(v_min, robot[3] - a_max * dt),
                      min(v_max, robot[3] + a_max * dt), 7)
    ws = np.linspace(max(w_min, robot[4] - alpha_max * dt),
                      min(w_max, robot[4] + alpha_max * dt), 11)
    best_c, best_v, best_w = np.inf, 0.0, 0.0
    for v in vs:
        for w in ws:
            c = cost(predict(robot, v, w), goal, obstacles)
            if c < best_c:
                best_c, best_v, best_w = c, v, w
    # COUNCIL FIX (pass 14, Allgöwer+Morari): emergency-stop fallback when
    # no trajectory in the dynamic window is collision-free. Production code
    # would also signal upstream for a global replan.
    if best_c == np.inf:
        best_v, best_w = 0.0, 0.0
        print(f"DWA: no admissible trajectory at step, e-stopping (signal replan upstream)")
    robot[0] += best_v * np.cos(robot[2]) * dt
    robot[1] += best_v * np.sin(robot[2]) * dt
    robot[2] += best_w * dt
    robot[3], robot[4] = best_v, best_w
    hist.append(robot[:2].copy())
    if np.linalg.norm(robot[:2] - goal) < 0.3:
        break
hist = np.array(hist)
print(f"Reached goal in {len(hist)} steps. Final distance: {np.linalg.norm(robot[:2] - goal):.3f} m")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(obstacles[:, 0], obstacles[:, 1], c='grey', s=400, label='Obstacles')
ax.plot(hist[:, 0], hist[:, 1], 'r-', lw=2, label='DWA trajectory')
ax.plot(0, 0, 'go', ms=14, label='Start')
ax.plot(*goal, 'b*', ms=18, label='Goal')
ax.set_aspect('equal'); ax.grid(); ax.legend()
ax.set_xlim(-1, 12); ax.set_ylim(-1, 12)
ax.set_title('Dynamic Window Approach Local Planner')
plt.tight_layout()
plt.show()


## References & rigor notes

**Complexity per control step.** $O(|V_d| \cdot T_p / \Delta t)$ trajectory simulations, where $|V_d|$ is the number of sampled $(v, \omega)$ pairs and $T_p / \Delta t$ is rollout length. In our notebook: $7 \times 11 \times 25 \approx 2000$ sims per step. Trivially parallelizable on GPU.

**Myopic failure mode.** DWA cannot reason about obstacles beyond the prediction horizon $T_p$. In non-convex obstacle fields (long alleys, dead ends) it gets stuck. Standard fix: pair DWA with a global planner (A\\*, RRT) — the global planner emits waypoints, DWA tracks them locally with reactivity.

**Convergence to goal** is *not* guaranteed by DWA alone. Lyapunov-based formulations exist (e.g., Brock & Khatib's *Global* DWA) that add a navigation-function term to guarantee progress.

**References.**
- Fox, D., Burgard, W., & Thrun, S. (1997). *The dynamic window approach to collision avoidance*. IEEE Robotics & Automation Magazine, 4(1), 23-33.
- Brock, O., & Khatib, O. (1999). *High-speed navigation using the global dynamic window approach*. ICRA 1999.
- Macenski, S., Singh, S., Martín, F., & Ginés, J. (2023). *Regulated pure pursuit for robot path tracking*. Autonomous Robots, 47(6).
